<a href="https://colab.research.google.com/github/nriveramell-ops/23f-desclasificado/blob/Brayant/Limpieza_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
## Instaling libraries
!pip install spacy
!pip install altair
!pip install sentence-transformers
!pip install gensim
!python -m spacy download es_dep_news_trf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 78.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.8/407.8 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.9/237.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 734.0/734.0 kB 41.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_dep_news_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
import pandas as pd
import spacy
import matplotlib.pyplot as plt
import numpy as np
import gensim
import nltk
from google.colab import drive
import pandas as pd
from spacy import displacy


nltk.download('punkt')
drive.mount('/content/drive')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Mounted at /content/drive


In [4]:

nlp = spacy.load("es_dep_news_trf")
import es_dep_news_trf
nlp = es_dep_news_trf.load()
doc = nlp("Esto es una frase.")
print([(w.text, w.pos_) for w in doc])

[('Esto', 'PRON'), ('es', 'AUX'), ('una', 'DET'), ('frase', 'NOUN'), ('.', 'PUNCT')]


In [5]:
# 3. Ruta de tu carpeta en Drive
RUTA_DRIVE = '/content/drive/MyDrive/Proyecto ML'

In [6]:
import os

def extraer_y_tokenizar(ruta):
    resultados = []

    for archivo in os.listdir(ruta):
        if archivo.endswith(".txt"):
            with open(os.path.join(ruta, archivo), 'r', encoding='utf-8') as f:
                texto = f.read()

            # Procesamiento con spaCy
            doc = nlp(texto)

            # Solo extracción de datos clave
            resultados.append({
                "archivo": archivo,
                "tokens": [t.text for t in doc],
                "entidades_persona": [ent.text for ent in doc.ents if ent.label_ == "PER"],
                "frases": [sent.text for sent in doc.sents]
            })
            print(f"Procesado: {archivo}")

    return resultados

# Ejecución
datos = extraer_y_tokenizar(RUTA_DRIVE)

Procesado: Vista_oral_281_del_Consejo_Supremo_de_Justicia_Militar_(20_de_febrero_de_1982).txt
Procesado: Vista_oral_281_del_Consejo_Supremo_de_Justicia_Militar_(22_de_febrero_de_1982).txt
Procesado: Vista_oral_281_del_Consejo_Supremo_de_Justicia_Militar_(24_de_febrero_de_1982).txt
Procesado: Vista_oral_281_del_Consejo_Supremo_de_Justicia_Militar_(25_de_febrero_de_1982).txt
Procesado: Vista_oral_281_del_Consejo_Supremo_de_Justicia_Militar_(26_de_febrero_de_1982).txt
Procesado: Información_integrada_(11_de_marzo_de_1982).txt
Procesado: Información_integrada_(16_de_marzo_de_1982).txt
Procesado: Vista_oral_281_del_Consejo_Supremo_de_Justicia_Militar_(2_de_marzo_de_1982).txt
Procesado: Vista_oral_281_del_Consejo_Supremo_de_Justicia_Militar_(5_de_marzo_de_1982).txt
Procesado: Vista_oral_281_del_Consejo_Supremo_de_Justicia_Militar_(8_de_marzo_de_1982).txt
Procesado: Vista_oral_281_del_Consejo_Supremo_de_Justicia_Militar_(9_de_marzo_de_1982).txt
Procesado: Vista_oral_281_del_Consejo_Supremo_

##Limpieza de los archivos


In [7]:

# 2. Obtener la lista oficial de stop words en español de spaCy
stop_words_espanol = nlp.Defaults.stop_words

# 3. ¡EL TRUCO ESTRELLA! Añadir palabras personalizadas que ensucian tus documentos del 23-F
palabras_extra_historicas = {"excmo", "sr", "d", "don", "doña", "folio", "num", "art", "cap", "gen"}
stop_words_espanol.update(palabras_extra_historicas)

# 4. Función específica para limpiar tus datos
def eliminar_stop_words(datos_extraidos):
    datos_sin_stopwords = []

    for documento in datos_extraidos:
        tokens_filtrados = []

        # Iteramos sobre los tokens que ya tienes extraídos
        for palabra in documento["tokens"]:
            # Pasamos a minúscula porque la lista de stop_words está en minúsculas
            palabra_min = palabra.lower()

            # Filtro: Si la palabra NO es una stop word, la conservamos
            if palabra_min not in stop_words_espanol:
                tokens_filtrados.append(palabra_min)

        # Copiamos los datos originales para no perder la información anterior
        doc_actualizado = documento.copy()

        # Añadimos la nueva lista limpia al diccionario
        doc_actualizado["tokens_sin_stopwords"] = tokens_filtrados

        datos_sin_stopwords.append(doc_actualizado)

    return datos_sin_stopwords

# 5. Ejecutar la función sobre tu variable 'datos'
datos_limpios = eliminar_stop_words(datos)

# Ver el resultado del primer documento
print("Tokens originales:", len(datos_limpios[0]["tokens"]))
print("Tokens tras borrar stop words:", len(datos_limpios[0]["tokens_sin_stopwords"]))

Tokens originales: 762
Tokens tras borrar stop words: 481


In [8]:
def eliminar_puntuacion(datos_limpios):
    datos_sin_puntuacion = []

    for documento in datos_limpios:
        tokens_filtrados = []

        # Iteramos sobre la lista que ya limpiamos en el paso anterior
        for palabra in documento["tokens_sin_stopwords"]:

            # EL ENFOQUE INTELIGENTE:
            # Le preguntamos al 'cerebro' de spaCy si esta palabra exacta es un signo de puntuación.
            # Si NO es puntuación (False), la guardamos.
            if not nlp.vocab[palabra].is_punct:
                tokens_filtrados.append(palabra)

        # Copiamos el diccionario para mantener el historial de transformaciones
        doc_actualizado = documento.copy()

        # Añadimos la nueva clave con el texto ultralimpio
        doc_actualizado["tokens_sin_puntuacion"] = tokens_filtrados

        datos_sin_puntuacion.append(doc_actualizado)

    return datos_sin_puntuacion

# Ejecutamos la función sobre la variable 'datos_limpios' que creamos antes
datos_finales = eliminar_puntuacion(datos_limpios)

# Comprobación de resultados
print("Tokens originales:", len(datos_finales[0]["tokens"]))
print("Tokens sin stop words:", len(datos_finales[0]["tokens_sin_stopwords"]))
print("Tokens listos (sin stop words ni puntuación):", len(datos_finales[0]["tokens_sin_puntuacion"]))

Tokens originales: 762
Tokens sin stop words: 481
Tokens listos (sin stop words ni puntuación): 372


In [9]:
def aplicar_lematizacion(datos_finales):
    datos_lematizados = []

    for documento in datos_finales:
        # 1. Unimos tus 372 tokens limpios en un solo texto separado por espacios.
        # Es mucho más eficiente para spaCy leer una frase larga que 372 palabras sueltas.
        texto_limpio = " ".join(documento["tokens_sin_puntuacion"])

        # 2. Pasamos el texto ya sin ruido por el modelo de spaCy
        doc_limpio = nlp(texto_limpio)

        # 3. Extraemos el lema (la raíz de diccionario) de cada token
        # Nota: Usamos token.lemma_ en lugar de token.text
        lemas = [token.lemma_ for token in doc_limpio]

        # 4. Copiamos el diccionario para no perder el historial
        doc_actualizado = documento.copy()

        # 5. Añadimos nuestra lista final lematizada
        doc_actualizado["tokens_lematizados"] = lemas
        datos_lematizados.append(doc_actualizado)

    return datos_lematizados

# Ejecutamos la función sobre los datos del paso anterior
datos_listos_para_analisis = aplicar_lematizacion(datos_finales)

# Vemos el 'Antes y Después' de las primeras 15 palabras
print("Tokens limpios (originales):", datos_finales[0]["tokens_sin_puntuacion"][:15])
print("Tokens lematizados (raíz):  ", datos_listos_para_analisis[0]["tokens_lematizados"][:15])

Tokens limpios (originales): ['c', 'sg/2820/20-02-82', '\n', 'dtor', '\n\n', 'nota', 'informativa', '\n\n', 'asunto', 'vista', 'oral', '2/81', '\n\n', '1.-', 'desarrollo']
Tokens lematizados (raíz):   ['c', 'sg/2820/20-02-82', '\n ', 'dtor', '\n\n ', 'nota', 'informativo', '\n\n ', 'asunto', 'vista', 'oral', '2/81', '\n\n ', '1.-', 'desarrollo']


In [ ]:
def filtrar_por_pos_tag(datos_listos_para_analisis, etiquetas_deseadas=["NOUN", "PROPN", "VERB"]):
    datos_con_pos_resultado = []

    for documento in datos_listos_para_analisis:
        # AQUÍ ESTABA EL ERROR: Cambiamos "datos_listos_para_analisis"
        # por la clave que contiene el texto (probablemente "texto")
        texto_a_procesar = documento.get("texto")

        # Si "texto" no funciona, probamos con "text" o reconstruimos de los tokens
        if not texto_a_procesar:
            texto_a_procesar = " ".join(documento["tokens"])

        doc_spacy = nlp(texto_a_procesar)

        tokens_filtrados_pos = []
        for token in doc_spacy:
            if token.pos_ in etiquetas_deseadas:
                tokens_filtrados_pos.append(f"{token.lemma_} ({token.pos_})")

        doc_actualizado = documento.copy()
        doc_actualizado["analisis_gramatical"] = tokens_filtrados_pos
        datos_con_pos_resultado.append(doc_actualizado)

    return datos_con_pos_resultado

# --- EJECUCIÓN ---
mis_etiquetas = ["PROPN", "VERB"]

# Asegúrate de que 'datos_listos_para_analisis' esté definida antes de esta línea
datos_finales_pos = filtrar_por_pos_tag(datos_listos_para_analisis, etiquetas_deseadas=mis_etiquetas)

print(f"Palabras detectadas con las etiquetas {mis_etiquetas}:")
print(datos_finales_pos[0]["analisis_gramatical"][:20])

In [ ]:
# --- NUEVA CELDA: DEPENDENCY PARSING ---

def analizar_dependencias(datos_finales_pos):
    """
    Analiza las relaciones sintácticas (quién hace qué)
    usando los nombres de variables de tu celda anterior.
    """
    datos_con_dependencias = []

    for documento in datos_finales_pos:
        # Recuperamos el texto original que usamos en el paso anterior
        # Si por algún motivo no existe "texto", lo reconstruye de los tokens
        texto_a_procesar = documento.get("texto", " ".join(documento.get("tokens", [])))

        doc_spacy = nlp(texto_a_procesar)

        # Creamos una lista para guardar las conexiones (Flechas)
        conexiones = []
        for token in doc_spacy:
            if not token.is_punct and not token.is_space:
                # Formato: Palabra --(Relación)--> Palabra de la que depende
                # Ejemplo: "Tejero" --(nsubj)--> "entrar"
                flecha = f"{token.text} --({token.dep_})--> {token.head.text}"
                conexiones.append(flecha)

        # Copiamos el diccionario y añadimos la nueva clave
        doc_actualizado = documento.copy()
        doc_actualizado["analisis_dependencias"] = conexiones
        datos_con_dependencias.append(doc_actualizado)

    return datos_con_dependencias

# --- EJECUCIÓN ---
# IMPORTANTE: Usamos 'datos_finales_pos' que es el nombre que definiste en tu celda anterior
datos_finales_dep = analizar_dependencias(datos_finales_pos)

# Ver el resultado para confirmar que no hay errores
print("✅ Análisis de dependencias completado con éxito.")
print(f"Ejemplo de conexiones en el primer documento:")
print(datos_finales_dep[0]["analisis_dependencias"][:15])

###AQUI DEBO CONTINUAR


In [ ]:
# --- NUEVA CELDA: NAMED ENTITY RECOGNITION (NER) ---

def analizar_entidades(datos_finales_dep):
    """
    Detecta y clasifica las Entidades Nombradas (personas, lugares, organizaciones)
    tomando como entrada los datos de la celda de dependencias anterior.
    """
    datos_con_ner = []

    for documento in datos_finales_dep:
        # Recuperamos el texto tal y como lo hiciste en la celda anterior
        texto_a_procesar = documento.get("texto", " ".join(documento.get("tokens", [])))

        doc_spacy = nlp(texto_a_procesar)

        # Creamos una lista para guardar las entidades encontradas
        entidades = []
        for ent in doc_spacy.ents:
            # Formato: Entidad -> ETIQUETA
            # Ejemplo: "Milans del Bosch" -> PER (Persona)
            # Ejemplo: "Congreso de los Diputados" -> ORG (Organización)
            info_entidad = f"{ent.text} -> {ent.label_}"
            entidades.append(info_entidad)

        # Copiamos el diccionario actual y añadimos la nueva clave para NER
        doc_actualizado = documento.copy()

        # Usamos list(set()) para eliminar duplicados si la misma persona aparece 10 veces
        doc_actualizado["entidades_nombradas"] = list(set(entidades))

        datos_con_ner.append(doc_actualizado)

    return datos_con_ner

# --- EJECUCIÓN ---
# IMPORTANTE: Usamos 'datos_finales_dep', la variable resultante de tu celda anterior
datos_finales_ner = analizar_entidades(datos_finales_dep)

# Ver el resultado para confirmar que todo funciona correctamente
print("✅ Reconocimiento de Entidades (NER) completado con éxito.")
print(f"Ejemplo de entidades únicas en el primer documento:")
print(datos_finales_ner[0]["entidades_nombradas"][:15])

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()
tfidf_tdm_sklearn = tfidf_vectorizer.fit_transform(datos_finales_ner)
vocabulary_sklearn = tfidf_vectorizer.get_feature_names_out()
print(vocabulary_sklearn)

AttributeError: 'dict' object has no attribute 'lower'

In [ ]:
datos_imp_palabras = pd.DataFrame(tfidf_tdm_sklearn.toarray().T, index=list(vocabulary_sklearn), columns=['d{}'.format(c) for c in range(1, len(datos_finales_ner)+1)])

In [ ]:
'''vectorizer = CountVectorizer(ngram_range=(1, 2), max_features = 500)
tf = vectorizer.fit_transform(df_inst['cleanText'])

print(vectorizer.vocabulary_)
print(vectorizer.get_feature_names_out())
print(tf.toarray().shape)

from wordcloud import WordCloud

sum_words = tf.sum(axis=0)
words = vectorizer.get_feature_names_out()
word_counts = np.asarray(sum_words).ravel().tolist()

word_dict = dict(zip(words, word_counts))

# Create a WordCloud object
wordcloud = WordCloud(width = 800, height = 800,
                background_color ='white',
                stopwords = None,
                min_font_size = 10).generate_from_frequencies(word_dict)

# Plot the WordCloud image
plt.figure(figsize = (8, 8), facecolor = None)
plt.imshow(wordcloud)
plt.axis("off")
plt.tight_layout(pad = 0)

plt.show()


word_dict_sorted = dict(sorted(word_dict.items(), key=lambda item: item[1], reverse=True))

# Unpack the words and frequencies into two separate lists
words, freqs = zip(*list(word_dict_sorted.items())[:40])

# Create a bar plot
plt.bar(words, freqs)
plt.xlabel('Words')
plt.ylabel('Frequency')
plt.title('Word Frequencies')
plt.xticks(rotation='vertical')
plt.show()  '''